In [10]:
#r "C:\Users\user\Desktop\Lukyanov\practice2026\ServerThreadTests\bin\Debug\net10.0\Task17.dll"
#r "C:\Users\user\Desktop\Lukyanov\practice2026\ServerThreadTests\bin\Debug\net10.0\ServerThreadTests.dll"

using System;
using System.Diagnostics;
using System.Linq;
using System.Threading;
using ScottPlot;
using task17;
using Task17;
using Task17.Tests;

public class HeavyCommand : ILongCommand
{
    private int _rem;
    public bool IsCompleted => _rem <= 0;
    public HeavyCommand(int n) => _rem = n;
    public void Execute()
    {
        if (!IsCompleted)
        {
            _rem--;
            double x = 0;
            for (int i = 0; i < 1000000; i++)
                x += Math.Sin(i) * Math.Cos(i);
        }
    }
}

public class SimpleScheduler : IScheduler
{
    private readonly Queue<ICommand> _q = new();
    public int Count => _q.Count;
    public void Add(ICommand c) { if (c != null) _q.Enqueue(c); }
    public bool HasCommand() => _q.Count > 0;
    public ICommand Select() => _q.Count > 0 ? _q.Dequeue() : null;
}

Console.WriteLine("Один поток / Round Robin\n");

var counts = new int[] { 1, 2, 4, 8, 16 };
var execPerCmd = 10;
var singleTimes = new double[counts.Length];
var rrTimes = new double[counts.Length];

for (int i = 0; i < counts.Length; i++)
{
    int n = counts[i];

    double singleTotal = 0;
    for (int run = 0; run < 3; run++)
    {
        var sw = Stopwatch.StartNew();
        for (int j = 0; j < n; j++)
        {
            var cmd = new HeavyCommand(execPerCmd);
            while (!cmd.IsCompleted) cmd.Execute();
        }
        sw.Stop();
        singleTotal += sw.Elapsed.TotalMilliseconds;
    }
    singleTimes[i] = singleTotal / 3;

    double rrTotal = 0;
    for (int run = 0; run < 3; run++)
    {
        var commands = new HeavyCommand[n];
        for (int j = 0; j < n; j++)
            commands[j] = new HeavyCommand(execPerCmd);

        var s = new SimpleScheduler();
        var st = new ServerThread(s);
        var sw = Stopwatch.StartNew();

        st.Start();
        for (int j = 0; j < n; j++)
            st.Add(commands[j]);

        while (!commands.All(c => c.IsCompleted))
            Thread.Sleep(1);

        sw.Stop();
        st.HardStop();
        st.Join();
        rrTotal += sw.Elapsed.TotalMilliseconds;
    }
    rrTimes[i] = rrTotal / 3;

    Console.WriteLine($"Команд: {n,2} | Один поток: {singleTimes[i],8:F1} мс | Round Robin: {rrTimes[i],8:F1} мс");
}

var plot = new Plot();
plot.Title("Время выполнения: Один поток vs Round Robin");
plot.XLabel("Количество команд");
plot.YLabel("Время (мс)");

plot.Add.Scatter(counts.Select(x => (double)x).ToArray(), singleTimes)
    .Label = "Один поток";
plot.Add.Scatter(counts.Select(x => (double)x).ToArray(), rrTimes)
    .Label = "Round Robin";

plot.ShowLegend();

var fn = "plot.png";
plot.SavePng(fn, 800, 600);
display(HTML($"<img src='{fn}?t={DateTime.Now.Ticks}' width='700'/>"));

Один поток / Round Robin

Команд:  1 | Один поток:    252.4 мс | Round Robin:    226.1 мс
Команд:  2 | Один поток:    498.0 мс | Round Robin:    482.3 мс
Команд:  4 | Один поток:    969.2 мс | Round Robin:    982.0 мс
Команд:  8 | Один поток:   1952.6 мс | Round Robin:   1898.7 мс
Команд: 16 | Один поток:   3818.8 мс | Round Robin:   3874.3 мс
